### Imports

In [ ]:
import pandas as pd
import ta
import yfinance as yf
import alpaca_trade_api as trade_api

### Get stock data

**Currently a test**

In [1]:
ticker = "AAPL"
data = yf.download(ticker, period="3mo", interval="1d")

# fix close data typing issue
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
print(type(data['Close']))
print(data['Close'].shape)
print(data.tail())

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.series.Series'>
(62,)
Price            Close        High         Low        Open    Volume
Date                                                                
2026-04-24  271.059998  273.059998  269.649994  272.760010  38157100
2026-04-27  267.609985  268.359985  265.070007  266.089996  41466800
2026-04-28  270.709991  273.230011  268.660004  272.339996  40018900
2026-04-29  270.170013  271.040009  267.040009  267.549988  30047900
2026-04-30  271.350006  275.940002  268.140015  270.500000  69274050


### Adding indicators

Right now I am using relative strength index (RSI). RSI is the relative strength index from 0-100. 30- = oversold (price drop), 70+ = overbought. I'm using RSI to detect potential rebounds after drops

Maybe I could use other indicators such as moving average, multiples such as Price to earnings, price to book value, + price to free cash flow

In [ ]:
data['rsi'] = ta.momentum.RSIIndicator(data['Close'], window=14).rsi()
data['rsi']

Date
2026-02-02          NaN
2026-02-03          NaN
2026-02-04          NaN
2026-02-05          NaN
2026-02-06          NaN
                ...    
2026-04-24    59.671863
2026-04-27    55.161877
2026-04-28    58.217676
2026-04-29    57.482801
2026-04-30    58.709387
Name: rsi, Length: 62, dtype: float64

### Detecting drops in price

In [11]:
data['pct_change'] = data['Close'].pct_change() * 100

def check_entry(row):
    return row['pct_change'] <= -3 or row['rsi'] < 30
data['buy_signal'] = data.apply(check_entry, axis=1)
data['buy_signal']

Date
2026-02-02    False
2026-02-03    False
2026-02-04    False
2026-02-05    False
2026-02-06    False
              ...  
2026-04-24    False
2026-04-27    False
2026-04-28    False
2026-04-29    False
2026-04-30    False
Name: buy_signal, Length: 62, dtype: bool

### Demonstrativ backtesting

**Currently spending 5% of balance on each trade**

BUY CONDITIONS:
- Buy signal present, no curent position
then uses all money t buy shares

SELL CONDITIONS:
- take profit @ +2%
- Stop loss @ -30%

In [ ]:
balance = 10000
total_shares = 0
total_cost = 0
risk_percent = 0.05

for i in range(len(data)):
    row = data.iloc[i]
    current_price = row['Close']

    # BUYING
    if row['buy_signal'] and total_shares == 0:
        entry_price = row['Close']
        trade_amount = balance * risk_percent
        shares = trade_amount / entry_price

        total_shares += shares
        total_cost += shares * entry_price
        balance -= shares * entry_price

        print(f"BUY at {entry_price}")

    # SCALING IN
    elif total_shares > 0:
        avg_entry_price = total_cost / total_shares

        drop_from_avg = (row['Close'] - avg_entry_price) / avg_entry_price
        if drop_from_avg <= -0.02 and balance > 0: # 2% lower
            trade_amount = balance * risk_percent
            shares = trade_amount / row['Close']

            total_shares += shares
            total_cost += shares * row['Close']

            balance -= shares * row['Close']

            print(f"Adding at {row['Close']}")

    # SELLING
    elif total_shares > 0:
        avg_entry_price = total_cost / total_shares
        change = (current_price - avg_entry_price) / avg_entry_price

        if change >= 0.02 or change <= -0.30:
            balance = total_shares * current_price
            total_shares = 0
            total_cost = 0
            print(f"SELL at {current_price}")